In [33]:
import pandas as pd
import numpy as np
import pennylane as qml
from scipy.linalg import sqrtm, inv
from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [34]:
X = pd.read_csv("../dataset/j_kampe.csv")
y = pd.read_csv("../dataset/distances.csv")["distance"]
random_forest_features = pd.read_csv("../results/random_forest_feature_selection.csv")["feature"].tolist()
X = X[random_forest_features]

X = X[1000:2000].values
y = y[1000:2000].to_numpy()

print(X.shape, y.shape)

(1000, 5) (1000,)


In [ ]:
n_qubits = 5
# dev_kernel = qml.device("lightning.qubit", wires=n_qubits)
# projector = np.zeros((2 ** n_qubits, 2 ** n_qubits))
# projector[0, 0] = 1


# @qml.qnode(dev_kernel)
# def angle_embedding_kernel(x1, x2):
#     qml.AngleEmbedding(x1, wires=range(n_qubits))
#     qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits))
#     return qml.expval(qml.Hermitian(projector, wires=range(n_qubits)))


# def angle_embedding_kernel_matrix(X):
#     return np.array([[angle_embedding_kernel(x1, x2) for x2 in X] for x1 in X])

def angle_embedding_kernel_matrix(X):
    """Angle Embedding kernel: k(x1,x2) = prod_i cos^2((x1_i - x2_i)/2)"""
    diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
    return np.prod(np.cos(diff / 2) ** 2, axis=2)


def rbf_kernel(x1, x2, gamma=1.0):
    """RBF kernel: k(x1,x2) = exp(-gamma * ||x1-x2||^2)"""
    return np.exp(-gamma * np.sum((x1 - x2) ** 2))


def rbf_kernel_matrix(X, gamma=1.0):
    """Compute the matrix whose entries are the RBF kernel
       evaluated on pairwise data from sets A and B."""
    return np.array([[rbf_kernel(x1, x2, gamma) for x2 in X] for x1 in X])


def pauli_feature_map_kernel(n_qubits, reps=2, entanglement="linear"):
    feature_map = pauli_feature_map(feature_dimension=n_qubits, reps=reps, entanglement=entanglement)
    sampler = StatevectorSampler()
    fidelity = ComputeUncompute(sampler=sampler)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)
    return quantum_kernel


def pauli_feature_map_kernel_matrix(X, n_qubits, reps=2, entanglement="linear"):
    quantum_kernel = pauli_feature_map_kernel(n_qubits, reps, entanglement)
    return quantum_kernel.evaluate(X)

In [36]:
def normalize_trace(K):
    return K * (K.shape[0] / np.trace(K))


def sqrt_psd(K, reg=1e-8):
    w, V = np.linalg.eigh(K)
    w = np.clip(w, reg, None)
    return V @ np.diag(np.sqrt(w)) @ V.T


def inv_psd(K, reg=1e-6):
    w, V = np.linalg.eigh(K)
    w = np.clip(w, reg, None)
    return V @ np.diag(1.0 / w) @ V.T


def geometric_difference(K1, K2, reg=1e-6):
    K2_sqrt = sqrt_psd(K2, reg)
    K1_inv = inv_psd(K1, reg)
    M = K2_sqrt @ K1_inv @ K2_sqrt
    spectral_norm = np.max(np.linalg.eigvalsh(M))
    return np.sqrt(np.abs(spectral_norm))


def model_complexity_s(K, y, reg=1e-6):
    K_reg = K + reg * np.eye(K.shape[0])
    K_inv = np.linalg.inv(K_reg)
    return float(y.T @ K_inv @ y)

In [37]:
def quantum_advantage_analysis(K_c, K_q, y, reg=1e-6):
    n = K_c.shape[0]
    
    g_cq = geometric_difference(K_c, K_q, reg)
    
    s_c = model_complexity_s(K_c, y, reg)
    s_q = model_complexity_s(K_q, y, reg)    
    s_c_bound = g_cq**2 * s_q
    
    print(f"N = {n}")
    print(f"sqrt(N) = {np.sqrt(n):.4f}")
    print(f"g_CQ = {g_cq:.6f}")
    print(f"s_C = {s_c:.6f}")
    print(f"s_Q = {s_q:.6f}")
    print(f"g_CQ^2 * s_Q = {s_c_bound:.6f}")
    print(f"s_C <= g_CQ^2 * s_Q: {s_c <= s_c_bound}")
    
    return {
        "g_cq": g_cq,
        "s_c": s_c,
        "s_q": s_q,
        "s_c_bound": s_c_bound
    }

# Angle Embedding Kernel vs RBF

In [38]:
K_c = normalize_trace(rbf_kernel_matrix(X, gamma=0.5))
# K_q_pauli = normalize_trace(pauli_feature_map_kernel_matrix(X, n_qubits, reps=2))
K_q_angle = normalize_trace(angle_embedding_kernel_matrix(X))

# print("Pauli Feature Map Kernel")
# results_pauli = quantum_advantage_analysis(K_c, K_q_pauli, y)

print("\nAngle Embedding Kernel")
results_angle = quantum_advantage_analysis(K_c, K_q_angle, y)


Angle Embedding Kernel
N = 1000
sqrt(N) = 31.6228
g_CQ = 3.402239
s_C = 691.127375
s_Q = 11800998.943789
g_CQ^2 * s_Q = 136599311.810842
s_C <= g_CQ^2 * s_Q: True
